In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, balanced_accuracy_score
import pandas as pd
import numpy as np
import math
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
import os
import pickle
from torchviz import make_dot

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error 
from scipy.stats import pearsonr
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA  # 添加PCA

# 检测GPU可用性
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. 数据准备
def load_data():
    # 加载组合响应数据
    combo_df = pd.read_csv('data/Comb/processed_combination_response.csv')
    
    # 对目标列进行1-处理
    combo_df['X/X0'] = 1 - combo_df['X/X0']
    combo_df['single_resp_1'] = 1 - combo_df['single_resp_1']
    combo_df['single_resp_2'] = 1 - combo_df['single_resp_2']
    
    # 加载药物特征
    drug_features = pd.read_csv('data/feature/Graphlet_features_6_standardized.csv')
    drug_features.set_index('name', inplace=True)
    
    # 加载细胞系特征
    cell_features = pd.read_csv('data/feature/cell_features_977d.csv')
    cell_features.set_index('Cell_Line', inplace=True)
    
    return combo_df, drug_features, cell_features

# 2. 数据预处理
def preprocess_data(combo_df, drug_features, cell_features):
    # 合并药物和细胞系特征
    data = []
    skipped_count = 0
    
    # 先收集所有药物特征用于PCA和scaler
    all_drug_feats = []
    all_cell_feats = []
    
    for _, row in combo_df.iterrows():
        drugA = row['drugA_name']
        drugB = row['drugB_name']
        cell_line = row['cell_line']
        
        # 只收集存在的特征
        if (drugA in drug_features.index and 
            drugB in drug_features.index and 
            cell_line in cell_features.index):
            all_drug_feats.append(drug_features.loc[drugA].values[2:])  # 跳过前3列
            all_drug_feats.append(drug_features.loc[drugB].values[2:])
            all_cell_feats.append(cell_features.loc[cell_line].values)
    
    # 正式处理数据
    for _, row in combo_df.iterrows():
        drugA = row['drugA_name']
        drugB = row['drugB_name']
        cell_line = row['cell_line']
        
        try:
            # 检查所有必需的特征是否存在
            if (drugA not in drug_features.index or 
                drugB not in drug_features.index or 
                cell_line not in cell_features.index):
                skipped_count += 1
                continue
                
            # 直接获取药物特征
            drugA_feat = drug_features.loc[drugA].values[2:].astype(np.float32)
            drugB_feat = drug_features.loc[drugB].values[2:].astype(np.float32)

            # 获取细胞系特征
            cell_feat = cell_features.loc[cell_line].values.astype(np.float32)

            # 组合特征
            combo_feat = np.concatenate([drugA_feat, drugB_feat, cell_feat]).astype(np.float32)
            
            # 获取目标值并确保为float32类型（已经是1-处理后的值）
            target = np.float32(row['X/X0'])
            
            data.append({
                'drugA_feat': drugA_feat,
                'drugB_feat': drugB_feat,
                'cell_feat': cell_feat,
                'combo_feat': combo_feat,
                'target': target,
                'drugA_name': drugA,
                'drugB_name': drugB,
                'cell_line': cell_line,
                'drugA_conc': np.float32(row['drugA Conc (µM)']),
                'drugB_conc': np.float32(row['drugB Conc (µM)']),
                'single_resp_1': np.float32(row['single_resp_1']),  # 已经是1-处理后的值
                'single_resp_2': np.float32(row['single_resp_2'])   # 已经是1-处理后的值
            })
        except Exception as e:
            print(f"Error processing sample {drugA}+{drugB} on {cell_line}: {str(e)}")
            skipped_count += 1
            continue
    
    print(f"Skipped {skipped_count} samples due to missing features or errors")
    print(f"Drug feature dimension: {len(data[0]['drugA_feat'])}")
    return pd.DataFrame(data)

# 3. 自定义数据集
class DrugCombinationDataset(Dataset):
    def __init__(self, data):
        self.data = data.reset_index(drop=True)  # 确保索引是连续的
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data.iloc[idx]
        return {
            'drugA_feat': torch.from_numpy(sample['drugA_feat']),
            'drugB_feat': torch.from_numpy(sample['drugB_feat']),
            'cell_feat': torch.from_numpy(sample['cell_feat']),
            'combo_feat': torch.from_numpy(sample['combo_feat']),
            'target': torch.tensor(sample['target'], dtype=torch.float32),
            'drugA_conc': torch.tensor(sample['drugA_conc'], dtype=torch.float32),
            'drugB_conc': torch.tensor(sample['drugB_conc'], dtype=torch.float32),
            'single_resp_1': torch.tensor(sample['single_resp_1'], dtype=torch.float32),
            'single_resp_2': torch.tensor(sample['single_resp_2'], dtype=torch.float32),
            'index': idx,  # 添加索引信息
            'drugA_name': sample['drugA_name'],
            'drugB_name': sample['drugB_name'],
            'cell_line': sample['cell_line']
        }

# 4. 神经网络模型 - 修改以确保alpha + beta = 1
class DrugCombinationModel(nn.Module):
    def __init__(self, drug_feat_dim=1000, cell_feat_dim=977):  # 修改默认药物特征维度为1000
        super(DrugCombinationModel, self).__init__()
        
        # 药物特征编码器
        self.drug_encoder = nn.Sequential(
            nn.Linear(drug_feat_dim, 1024),
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        
        # 细胞系特征编码器
        self.cell_encoder = nn.Sequential(
            nn.Linear(cell_feat_dim, 512),
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            # nn.Dropout(0.3)
        )
        
        # 药物A特征处理网络 (计算α) - 加入药物A剂量
        self.alpha_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 512),  # +1 for drugA_conc
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
        
        # 药物B特征处理网络 (计算β) - 加入药物B剂量
        self.beta_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 512),  # +1 for drugB_conc
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
        
        # 联合特征处理网络 (计算γ) - 加入两个药物剂量
        self.gamma_net = nn.Sequential(
            nn.Linear(256 + 256 + 256 + 2, 1024),  # +2 for drugA_conc and drugB_conc
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            # nn.Dropout(0.3),
            nn.Linear(512, 1)
        )
        
    def forward(self, drugA_feat, drugB_feat, cell_feat, combo_feat, drugA_conc, drugB_conc):
        # 编码药物特征和细胞特征
        encoded_drugA = self.drug_encoder(drugA_feat)
        encoded_drugB = self.drug_encoder(drugB_feat)
        encoded_cell = self.cell_encoder(cell_feat)
        
        # 计算α - 加入药物A剂量
        alpha_input = torch.cat([encoded_drugA, encoded_cell, drugA_conc.unsqueeze(1)], dim=1)
        alpha_raw = self.alpha_net(alpha_input)
        
        # 计算β - 加入药物B剂量
        beta_input = torch.cat([encoded_drugB, encoded_cell, drugB_conc.unsqueeze(1)], dim=1)
        beta_raw = self.beta_net(beta_input)
        
        # 计算γ - 加入两个药物剂量
        gamma_input = torch.cat([encoded_drugA, encoded_drugB, encoded_cell, 
                               drugA_conc.unsqueeze(1), drugB_conc.unsqueeze(1)], dim=1)
        gamma = self.gamma_net(gamma_input)

        # 确保alpha + beta = 1
        alpha_beta_sum = alpha_raw + beta_raw
        alpha = alpha_raw / alpha_beta_sum
        beta = beta_raw / alpha_beta_sum

        return alpha.squeeze(-1), beta.squeeze(-1), gamma.squeeze(-1)

# 5. 训练函数 - 修改以保存真实值和预测值
def train_model(model, train_loader, val_loader, train_df, val_df, epochs=100, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    model = model.to(device)
    
    train_losses = []
    val_losses = []
    
    # 用于保存所有epoch的α, β, γ值、真实值和预测值
    all_alpha_beta_gamma = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        # 保存当前epoch的训练集α, β, γ值
        epoch_train_results = []
        
        for batch in train_loader:
            drugA_feat = batch['drugA_feat'].to(device)
            drugB_feat = batch['drugB_feat'].to(device)
            cell_feat = batch['cell_feat'].to(device)
            combo_feat = batch['combo_feat'].to(device)
            target = batch['target'].to(device)
            drugA_conc = batch['drugA_conc'].to(device)
            drugB_conc = batch['drugB_conc'].to(device)
            single_resp_1 = batch['single_resp_1'].to(device)
            single_resp_2 = batch['single_resp_2'].to(device)
            indices = batch['index'].numpy()
            
            optimizer.zero_grad()
            
            alpha, beta, gamma = model(drugA_feat, drugB_feat, cell_feat, combo_feat, drugA_conc, drugB_conc)
            # 使用处理后的single_resp_1和single_resp_2进行预测
            pred = alpha * single_resp_1 + beta * single_resp_2 + gamma
            
            loss = criterion(pred, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * drugA_feat.size(0)
            
            # 保存当前batch的α, β, γ值、真实值和预测值
            alpha_np = alpha.cpu().detach().numpy()
            beta_np = beta.cpu().detach().numpy()
            gamma_np = gamma.cpu().detach().numpy()
            pred_np = pred.cpu().detach().numpy()
            target_np = target.cpu().detach().numpy()
            
            for j, idx in enumerate(indices):
                sample_data = train_df.iloc[idx]
                epoch_train_results.append({
                    'epoch': epoch + 1,
                    'drugA_name': sample_data['drugA_name'],
                    'drugB_name': sample_data['drugB_name'],
                    'cell_line': sample_data['cell_line'],
                    'drugA_conc': sample_data['drugA_conc'],
                    'drugB_conc': sample_data['drugB_conc'],
                    'target': target_np[j],  # 真实值
                    'prediction': pred_np[j],  # 预测值
                    'single_resp_1': sample_data['single_resp_1'],
                    'single_resp_2': sample_data['single_resp_2'],
                    'alpha': alpha_np[j],
                    'beta': beta_np[j],
                    'gamma': gamma_np[j],
                    'set': 'train'
                })
        
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)
        
        # 验证集评估
        val_loss, val_metrics, val_results = evaluate_model(model, val_loader, val_df, epoch + 1)
        val_losses.append(val_loss)
        
        # 合并训练集和验证集结果
        epoch_results = epoch_train_results + val_results
        all_alpha_beta_gamma.extend(epoch_results)
        
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
        print(f'Val Metrics - MSE: {val_metrics["mse"]:.4f}, RMSE: {val_metrics["rmse"]:.4f}, '
              f'MAE: {val_metrics["mae"]:.4f}, PCC: {val_metrics["pcc"]:.4f}, R2: {val_metrics["r2"]:.4f}')  
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model_1_minus.pth')
    
    # 保存所有epoch的α, β, γ值、真实值和预测值到CSV
    results_df = pd.DataFrame(all_alpha_beta_gamma)
    results_df.to_csv('alpha_beta_gamma_training_history.csv', index=False)
    print("Saved alpha, beta, gamma values and predictions for all epochs to alpha_beta_gamma_training_history.csv")
    
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, epochs+1), train_losses, label='Train Loss')
    plt.plot(range(1, epochs+1), val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss (1-X/X0)')
    plt.legend()
    plt.grid(True)
    plt.savefig('loss_curve_1_minus.png')
    plt.close()
    
    model.load_state_dict(torch.load('best_model_1_minus.pth', map_location=device))
    return model, train_losses, val_losses

# 6. 评估函数 - 修改以返回真实值和预测值
def evaluate_model(model, data_loader, data_df, epoch):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    results = []
    
    with torch.no_grad():
        for batch in data_loader:
            drugA_feat = batch['drugA_feat'].to(device)
            drugB_feat = batch['drugB_feat'].to(device)
            cell_feat = batch['cell_feat'].to(device)
            combo_feat = batch['combo_feat'].to(device)
            target = batch['target'].to(device)
            drugA_conc = batch['drugA_conc'].to(device)
            drugB_conc = batch['drugB_conc'].to(device)
            single_resp_1 = batch['single_resp_1'].to(device)
            single_resp_2 = batch['single_resp_2'].to(device)
            indices = batch['index'].numpy()
            
            alpha, beta, gamma = model(drugA_feat, drugB_feat, cell_feat, combo_feat, drugA_conc, drugB_conc)
            # 使用处理后的single_resp_1和single_resp_2进行预测
            pred = alpha * single_resp_1 + beta * single_resp_2 + gamma
            
            all_preds.extend(pred.cpu().numpy().flatten())
            all_targets.extend(target.cpu().numpy().flatten())
            
            loss = nn.MSELoss()(pred, target)
            total_loss += loss.item() * drugA_feat.size(0)
            
            # 保存当前batch的α, β, γ值、真实值和预测值
            alpha_np = alpha.cpu().numpy()
            beta_np = beta.cpu().numpy()
            gamma_np = gamma.cpu().numpy()
            pred_np = pred.cpu().numpy()
            target_np = target.cpu().numpy()
            
            for j, idx in enumerate(indices):
                sample_data = data_df.iloc[idx]
                results.append({
                    'epoch': epoch,
                    'drugA_name': sample_data['drugA_name'],
                    'drugB_name': sample_data['drugB_name'],
                    'cell_line': sample_data['cell_line'],
                    'drugA_conc': sample_data['drugA_conc'],
                    'drugB_conc': sample_data['drugB_conc'],
                    'target': target_np[j],  # 真实值
                    'prediction': pred_np[j],  # 预测值
                    'single_resp_1': sample_data['single_resp_1'],
                    'single_resp_2': sample_data['single_resp_2'],
                    'alpha': alpha_np[j],
                    'beta': beta_np[j],
                    'gamma': gamma_np[j],
                    'set': 'validation'
                })
    
    total_loss /= len(data_loader.dataset)
    
    assert len(all_preds) == len(all_targets), f"预测值数量({len(all_preds)})与真实值数量({len(all_targets)})不匹配"
    
    mse = mean_squared_error(all_targets, all_preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(all_targets, all_preds)
    pcc, _ = pearsonr(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)
    
    metrics = {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'pcc': pcc,
        'r2': r2
    }
    
    return total_loss, metrics, results

# 7. 保存最终α, β, γ值、真实值和预测值的函数
def save_final_alpha_beta_gamma(model, data_loader, data_df):
    model.eval()
    results = []
    
    with torch.no_grad():
        for batch in data_loader:
            # 获取batch中的索引
            indices = batch['index'].numpy()
            batch_data = data_df.iloc[indices]
            
            drugA_feat = batch['drugA_feat'].to(device)
            drugB_feat = batch['drugB_feat'].to(device)
            cell_feat = batch['cell_feat'].to(device)
            combo_feat = batch['combo_feat'].to(device)
            drugA_conc = batch['drugA_conc'].to(device)
            drugB_conc = batch['drugB_conc'].to(device)
            target = batch['target'].to(device)
            single_resp_1 = batch['single_resp_1'].to(device)
            single_resp_2 = batch['single_resp_2'].to(device)
            
            alpha, beta, gamma = model(drugA_feat, drugB_feat, cell_feat, combo_feat, drugA_conc, drugB_conc)
            pred = alpha * single_resp_1 + beta * single_resp_2 + gamma
            
            alpha = alpha.cpu().numpy().flatten()
            beta = beta.cpu().numpy().flatten()
            gamma = gamma.cpu().numpy().flatten()
            pred_np = pred.cpu().numpy().flatten()
            target_np = target.cpu().numpy().flatten()
            
            for j in range(len(alpha)):
                sample_data = batch_data.iloc[j]
                results.append({
                    'drugA_name': sample_data['drugA_name'],
                    'drugB_name': sample_data['drugB_name'],
                    'cell_line': sample_data['cell_line'],
                    'drugA_conc': sample_data['drugA_conc'],
                    'drugB_conc': sample_data['drugB_conc'],
                    'target': target_np[j],  # 真实值
                    'prediction': pred_np[j],  # 预测值
                    'single_resp_1': sample_data['single_resp_1'],  # 已经是1-处理后的值
                    'single_resp_2': sample_data['single_resp_2'],  # 已经是1-处理后的值
                    'alpha': alpha[j],
                    'beta': beta[j],
                    'gamma': gamma[j]
                })
    
    results_df = pd.DataFrame(results)
    column_order = [
        'drugA_name', 'drugB_name', 'cell_line',
        'drugA_conc', 'drugB_conc', 
        'target', 'prediction',  # 真实值和预测值
        'single_resp_1', 'single_resp_2',
        'alpha', 'beta', 'gamma'
    ]
    results_df = results_df[column_order]
    results_df.to_csv('final_alpha_beta_gamma_values_1_minus.csv', index=False)
    print("Saved final alpha, beta, gamma values with drug and cell line info (1-X/X0)")

# 8. 主函数
def main():
    combo_df, drug_features, cell_features = load_data()
    data_df = preprocess_data(combo_df, drug_features, cell_features)
    
    if len(data_df) == 0:
        print("No valid data samples after preprocessing. Exiting.")
        return
    
    sample = data_df.iloc[0]
    drug_feat_dim = len(sample['drugA_feat'])
    cell_feat_dim = len(sample['cell_feat'])
    print(f"Drug feature dimension: {drug_feat_dim}")
    print(f"Cell feature dimension: {cell_feat_dim}")
    
    train_df, test_df = train_test_split(data_df, test_size=0.2, random_state=42)
    
    train_dataset = DrugCombinationDataset(train_df)
    test_dataset = DrugCombinationDataset(test_df)
    print(f"Train dataset size: {len(train_dataset)}") 
    print(f"Test dataset size: {len(test_dataset)}")
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    model = DrugCombinationModel(drug_feat_dim=drug_feat_dim, cell_feat_dim=cell_feat_dim)
    
    trained_model, train_losses, val_losses = train_model(
        model, train_loader, test_loader, train_df, test_df, epochs=100, lr=0.0001
    )
    
    final_loss, final_metrics, _ = evaluate_model(trained_model, test_loader, test_df, 100)
    print(f'\nFinal Test Metrics (1-X/X0):')
    print(f'MSE: {final_metrics["mse"]:.4f}')
    print(f'RMSE: {final_metrics["rmse"]:.4f}')
    print(f'MAE: {final_metrics["mae"]:.4f}') 
    print(f'PCC: {final_metrics["pcc"]:.4f}')
    print(f'R2: {final_metrics["r2"]:.4f}')
    
    torch.save(trained_model.state_dict(), 'final_model_1_minus.pth')
    save_final_alpha_beta_gamma(trained_model, test_loader, test_df)

if __name__ == '__main__':
    main()

Using device: cuda
Skipped 2304 samples due to missing features or errors
Drug feature dimension: 6890
Drug feature dimension: 6890
Cell feature dimension: 977
Train dataset size: 24448
Test dataset size: 6112
Epoch 1/100, Train Loss: 0.0144, Val Loss: 0.0124
Val Metrics - MSE: 0.0124, RMSE: 0.1113, MAE: 0.0810, PCC: 0.9075, R2: 0.8180
Epoch 2/100, Train Loss: 0.0121, Val Loss: 0.0113
Val Metrics - MSE: 0.0113, RMSE: 0.1065, MAE: 0.0785, PCC: 0.9162, R2: 0.8333
Epoch 3/100, Train Loss: 0.0112, Val Loss: 0.0099
Val Metrics - MSE: 0.0099, RMSE: 0.0996, MAE: 0.0727, PCC: 0.9246, R2: 0.8542
Epoch 4/100, Train Loss: 0.0101, Val Loss: 0.0104
Val Metrics - MSE: 0.0104, RMSE: 0.1020, MAE: 0.0763, PCC: 0.9236, R2: 0.8469
Epoch 5/100, Train Loss: 0.0092, Val Loss: 0.0097
Val Metrics - MSE: 0.0097, RMSE: 0.0987, MAE: 0.0719, PCC: 0.9283, R2: 0.8569
Epoch 6/100, Train Loss: 0.0084, Val Loss: 0.0081
Val Metrics - MSE: 0.0081, RMSE: 0.0902, MAE: 0.0658, PCC: 0.9398, R2: 0.8803
Epoch 7/100, Train Los

C:\Users\HXLS-PC\AppData\Local\Temp\ipykernel_3088\321586140.py:332: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_1_minus.pth',


Final Test Metrics (1-X/X0):
MSE: 0.0043
RMSE: 0.0653
MAE: 0.0462
PCC: 0.9681
R2: 0.9372
Saved final alpha, beta, gamma values with drug and cell line info (1-X/X0)
